# 🧠 Heartbreak Severity Classifier — Test Model
---
Notebook ini bisa dijalankan di **Google Colab maupun Jupyter lokal**.

**Input yang dibutuhkan:**
- Data Demografis: Nama, Umur, Jenis Kelamin, Lama Hubungan, Lama Sejak Putus
- Data Psikometri: Score B, C, D, E, F, G, H (skala 1-5)

**Output:** Prediksi tingkat heartbreak → `Ringan` / `Sedang` / `Berat`

In [1]:
# ============================================================
# CELL 0: SETUP — DETEKSI ENVIRONMENT & LOAD FILE .PKL
# Jalankan salah satu blok (A atau B) sesuai kebutuhan
# ============================================================
import os

# Cek apakah sedang berjalan di Google Colab
try:
    import google.colab
    IS_COLAB = True
    print('✅ Environment: Google Colab')
except ImportError:
    IS_COLAB = False
    print('✅ Environment: Jupyter Lokal')

print()
print('Pilih cara load file .pkl:')
print('  ▶ CARA A: Mount Google Drive  (jalankan Cell A)')
print('  ▶ CARA B: Upload file manual  (jalankan Cell B)')
print('  ▶ CARA C: File sudah ada lokal (langsung ke Cell 1)')

✅ Environment: Google Colab

Pilih cara load file .pkl:
  ▶ CARA A: Mount Google Drive  (jalankan Cell A)
  ▶ CARA B: Upload file manual  (jalankan Cell B)
  ▶ CARA C: File sudah ada lokal (langsung ke Cell 1)


In [2]:
# ============================================================
# CELL A: CARA A — MOUNT GOOGLE DRIVE
# Gunakan jika file .pkl ada di Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ⬇️ Ganti path sesuai lokasi file .pkl di Google Drive kamu
BUNDLE_PATH = '/content/drive/MyDrive/ANN/Artificial Neural Network/heartbreak_classifier_bundle.pkl'

print(f'Path file: {BUNDLE_PATH}')
print(f'File exists: {os.path.exists(BUNDLE_PATH)}')

Mounted at /content/drive
Path file: /content/drive/MyDrive/ANN/Artificial Neural Network/heartbreak_classifier_bundle.pkl
File exists: True


In [3]:
# ============================================================
# CELL B: CARA B — UPLOAD FILE LANGSUNG KE COLAB SESSION
# Gunakan jika file .pkl ada di komputer lokal
# ============================================================
from google.colab import files

print('Pilih file heartbreak_classifier_bundle.pkl dari komputer kamu...')
uploaded = files.upload()

# Otomatis ambil nama file yang diupload
BUNDLE_PATH = list(uploaded.keys())[0]
print(f'\n✅ File berhasil diupload: {BUNDLE_PATH}')

Pilih file heartbreak_classifier_bundle.pkl dari komputer kamu...


Saving heartbreak_classifier_bundle.pkl to heartbreak_classifier_bundle.pkl

✅ File berhasil diupload: heartbreak_classifier_bundle.pkl


In [4]:
# ============================================================
# CELL C: CARA C — FILE SUDAH ADA DI FOLDER LOKAL
# Gunakan jika run di Jupyter lokal & file .pkl satu folder
# ============================================================
BUNDLE_PATH = 'heartbreak_classifier_bundle.pkl'
print(f'Path file: {BUNDLE_PATH}')

Path file: heartbreak_classifier_bundle.pkl


In [18]:
# ============================================================
# CELL 1: IMPORT & LOAD MODEL
# ============================================================
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    bundle = joblib.load(BUNDLE_PATH)
    model         = bundle['model']
    scaler        = bundle['scaler']
    decode_label  = bundle['decode_label']
    feature_names = bundle['feature_names']
    print('✅ Model berhasil dimuat!')
    print(f'   Total fitur : {len(feature_names)}')
    print(f'   Output label: {list(decode_label.values())}')
except FileNotFoundError:
    print(f'❌ File tidak ditemukan: {BUNDLE_PATH}')
except NameError:
    print('❌ BUNDLE_PATH belum di-set. Jalankan Cell A, B, atau C dulu.')

✅ Model berhasil dimuat!
   Total fitur : 39
   Output label: ['Ringan', 'Sedang', 'Berat']


In [19]:
# ============================================================
# CELL DIAGNOSA: CEK FEATURE NAMES YANG SEBENARNYA DISIMPAN
# Jalankan sekali untuk melihat isi bundle vs scaler
# ============================================================

# Ambil feature names yang BENAR-BENAR dipakai scaler saat training
TRUE_FEATURE_NAMES = list(scaler.feature_names_in_)

print(f'Feature names di bundle   : {len(feature_names)} kolom')
print(f'Feature names di scaler   : {len(TRUE_FEATURE_NAMES)} kolom')
print()

# Tampilkan kolom yang berkaitan dengan < atau >
print('Kolom di BUNDLE (Lama / Sudah):')
for f in feature_names:
    if 'Lama' in f or 'Sudah' in f:
        print(f'  {repr(f)}')

print()
print('Kolom di SCALER (Lama / Sudah):')
for f in TRUE_FEATURE_NAMES:
    if 'Lama' in f or 'Sudah' in f:
        print(f'  {repr(f)}')

print()
# Cek apakah ada perbedaan
diff = set(TRUE_FEATURE_NAMES) - set(feature_names)
if diff:
    print(f'⚠️  Ada {len(diff)} kolom di scaler yang TIDAK ada di bundle feature_names:')
    for d in sorted(diff):
        print(f'   {repr(d)}')
else:
    print('✅ Bundle dan scaler konsisten!')

# Gunakan scaler.feature_names_in_ sebagai referensi utama
print()
print('✅ TRUE_FEATURE_NAMES siap digunakan (dari scaler).')

Feature names di bundle   : 39 kolom
Feature names di scaler   : 39 kolom

Kolom di BUNDLE (Lama / Sudah):
  'Lama Hubungan Sebelum Putus_3 - 5 tahun'
  'Lama Hubungan Sebelum Putus_6 bulan - 1 tahun'
  'Lama Hubungan Sebelum Putus_ 6 bulan'
  'Lama Hubungan Sebelum Putus_ 5 tahun'
  'Sudah Berapa Lama Sejak Putus?_3 - 6 bulan'
  'Sudah Berapa Lama Sejak Putus?_6 - 12 bulan'
  'Sudah Berapa Lama Sejak Putus?_ 1 bulan'
  'Sudah Berapa Lama Sejak Putus?_ 1 tahun'

Kolom di SCALER (Lama / Sudah):
  'Lama Hubungan Sebelum Putus_3 - 5 tahun'
  'Lama Hubungan Sebelum Putus_6 bulan - 1 tahun'
  'Lama Hubungan Sebelum Putus_< 6 bulan'
  'Lama Hubungan Sebelum Putus_> 5 tahun'
  'Sudah Berapa Lama Sejak Putus?_3 - 6 bulan'
  'Sudah Berapa Lama Sejak Putus?_6 - 12 bulan'
  'Sudah Berapa Lama Sejak Putus?_< 1 bulan'
  'Sudah Berapa Lama Sejak Putus?_> 1 tahun'

⚠️  Ada 4 kolom di scaler yang TIDAK ada di bundle feature_names:
   'Lama Hubungan Sebelum Putus_< 6 bulan'
   'Lama Hubungan Sebelum Pu

In [20]:
# ============================================================
# CELL 2: FUNCTION PREDICT & TAMPILKAN HASIL
# Menggunakan TRUE_FEATURE_NAMES dari scaler — pasti cocok 100%
# ============================================================

def predict_heartbreak(
    umur, jenis_kelamin, pendidikan,
    lama_hubungan, sudah_berapa_lama_putus,
    siapa_yang_mengakhiri, masih_komunikasi, frekuensi_medsos_mantan,
    score_B, score_C, score_D, score_E, score_F, score_G, score_H
):
    """
    Encoding manual — kolom diambil langsung dari scaler.feature_names_in_
    sehingga selalu cocok dengan apa yang dipakai saat training.
    """

    # Buat baris kosong berdasarkan feature names DARI SCALER (ground truth)
    row = {col: 0 for col in TRUE_FEATURE_NAMES}

    # Isi fitur numerik
    row['Umur'] = umur

    # Helper: set one-hot kolom yang sesuai
    def set_onehot(prefix, value):
        col = f"{prefix}_{value}"
        if col in row:
            row[col] = 1
        else:
            # Coba juga tanpa spasi ekstra (fallback)
            col2 = f"{prefix}_{value.strip()}"
            if col2 in row:
                row[col2] = 1

    # Isi one-hot per fitur kategorik
    set_onehot('Jenis Kelamin',                               jenis_kelamin)
    set_onehot('Pendidikan',                                  pendidikan)
    set_onehot('Lama Hubungan Sebelum Putus',                 lama_hubungan)
    set_onehot('Sudah Berapa Lama Sejak Putus?',              sudah_berapa_lama_putus)
    set_onehot('Siapa yang Mengakhiri Hubungan?',             siapa_yang_mengakhiri)
    set_onehot('Apakah Masih Berkomunikasi dengan Mantan?',   masih_komunikasi)
    set_onehot('Seberapa Sering Melihat Media Sosial Mantan?',frekuensi_medsos_mantan)

    # Isi psikometri
    row['score_B']          = score_B
    row['score_C']          = score_C
    row['score_D']          = score_D
    row['score_E']          = score_E
    row['score_F_reversed'] = 6 - score_F
    row['score_G']          = score_G
    row['score_H_reversed'] = 6 - score_H

    # Buat DataFrame dengan urutan kolom yang benar
    X_input = pd.DataFrame([row])[TRUE_FEATURE_NAMES]

    # Scale & Predict
    X_scaled  = scaler.transform(X_input)
    pred_idx  = model.predict(X_scaled)[0]
    pred_prob = model.predict_proba(X_scaled)[0]

    return {
        'severity':    decode_label[pred_idx],
        'probability': {
            'Ringan': float(pred_prob[0]),
            'Sedang': float(pred_prob[1]) if len(pred_prob) > 1 else 0.0,
            'Berat':  float(pred_prob[2]) if len(pred_prob) > 2 else 0.0
        },
        'confidence': float(pred_prob.max())
    }


def tampilkan_hasil(nama, result):
    emoji_map = {'Ringan': '🟢', 'Sedang': '🟡', 'Berat': '🔴'}
    sev  = result['severity']
    conf = result['confidence']
    print('=' * 55)
    print(f'  HASIL PREDIKSI HEARTBREAK — {nama.upper()}')
    print('=' * 55)
    print(f'  Tingkat Heartbreak : {emoji_map.get(sev, "⚪")}  {sev}')
    print(f'  Confidence         : {conf*100:.2f}%')
    print()
    print('  Probabilitas per Kategori:')
    for kat, prob in result['probability'].items():
        bar = '█' * int(prob * 30) + '░' * (30 - int(prob * 30))
        em = emoji_map.get(kat, '⚪')
        print(f'  {em} {kat:6s} [{bar}] {prob*100:6.2f}%')
    print('=' * 55)


print('✅ Fungsi prediksi siap digunakan!')

✅ Fungsi prediksi siap digunakan!


---
## 📝 Input Data — Isi di bawah ini

| Field | Pilihan Nilai |
|---|---|
| `jenis_kelamin` | `'Laki-laki'` \| `'Perempuan'` \| `'Lainnya'` |
| `pendidikan` | `'SMP/Sederajat'` \| `'SMA/Sederajat'` \| `'Diploma (D1/D2/D3)'` \| `'S1'` \| `'S2'` \| `'S3'` \| `'Lainnya'` |
| `lama_hubungan` | `'< 6 bulan'` \| `'6 bulan - 1 tahun'` \| `'1 - 3 tahun'` \| `'3 - 5 tahun'` \| `'> 5 tahun'` |
| `lama_sejak_putus` | `'< 1 bulan'` \| `'1 - 3 bulan'` \| `'3 - 6 bulan'` \| `'6 - 12 bulan'` \| `'> 1 tahun'` |
| `siapa_yang_mengakhiri` | `'Saya yang mengakhiri'` \| `'Pasangan yang mengakhiri'` \| `'Tidak jelas'` |
| `masih_komunikasi` | `'Tidak pernah'` \| `'Kadang-kadang'` \| `'Sering'` \| `'Setiap hari'` |
| `frekuensi_medsos` | `'Tidak pernah'` \| `'Jarang'` \| `'Kadang-kadang'` \| `'Sekali sehari'` \| `'Hampir setiap hari'` \| `'Beberapa kali sehari'` |
| `score_B` s/d `score_H` | `1` sampai `5` (boleh desimal, misal `3.5`) |

In [21]:
# ============================================================
# CELL 3: ✏️ ISI INPUT DI SINI — SATU ORANG
# ============================================================

# --- DATA PRIBADI ---
nama             = "Budi Santoso"           # Nama (hanya untuk tampilan output)
umur             = 25                       # Umur dalam tahun
jenis_kelamin    = 'Perempuan'              # Laki-laki / Perempuan / Lainnya
lama_hubungan    = '> 5 tahun'              # Lama hubungan sebelum putus
lama_sejak_putus = '6 - 12 bulan'          # Sudah berapa lama sejak putus

# --- DATA TAMBAHAN ---
pendidikan             = 'S1'
siapa_yang_mengakhiri  = 'Pasangan yang mengakhiri'
masih_komunikasi       = 'Kadang-kadang'
frekuensi_medsos       = 'Beberapa kali sehari'

# --- PSIKOMETRI SCORES (skala 1-5) ---
# B = Ruminasi (terus memikirkan mantan)
# C = Identitas diri pasca putus
# D = Depresi / kesedihan
# E = Kesepian
# F = Penerimaan (reversed: nilai rendah = belum terima)
# G = Keinginan untuk kembali bersama mantan
# H = Resiliensi (reversed: nilai rendah = tidak resilient)
score_B = 4.0
score_C = 3.0
score_D = 4.5
score_E = 3.5
score_F = 2.0
score_G = 4.0
score_H = 2.5

# ============================================================
result = predict_heartbreak(
    umur=umur, jenis_kelamin=jenis_kelamin, pendidikan=pendidikan,
    lama_hubungan=lama_hubungan, sudah_berapa_lama_putus=lama_sejak_putus,
    siapa_yang_mengakhiri=siapa_yang_mengakhiri,
    masih_komunikasi=masih_komunikasi,
    frekuensi_medsos_mantan=frekuensi_medsos,
    score_B=score_B, score_C=score_C, score_D=score_D,
    score_E=score_E, score_F=score_F, score_G=score_G, score_H=score_H
)

tampilkan_hasil(nama, result)

  HASIL PREDIKSI HEARTBREAK — BUDI SANTOSO
  Tingkat Heartbreak : 🟢  Ringan
  Confidence         : 66.72%

  Probabilitas per Kategori:
  🟢 Ringan [████████████████████░░░░░░░░░░]  66.72%
  🟡 Sedang [█████████░░░░░░░░░░░░░░░░░░░░░]  33.28%
  🔴 Berat  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0.00%


---
## 🔁 Test Multiple Kasus Sekaligus

In [22]:
# ============================================================
# CELL 4: BATCH TEST — BEBERAPA KASUS SEKALIGUS
# ============================================================

test_cases = [
    {
        'nama': 'Siti Rahayu', 'umur': 22,
        'jenis_kelamin': 'Perempuan', 'pendidikan': 'S1',
        'lama_hubungan': '< 6 bulan', 'lama_sejak_putus': '< 1 bulan',
        'siapa_yang_mengakhiri': 'Pasangan yang mengakhiri',
        'masih_komunikasi': 'Setiap hari', 'frekuensi_medsos': 'Beberapa kali sehari',
        'score_B':5.0,'score_C':1.5,'score_D':5.0,'score_E':4.5,'score_F':1.5,'score_G':5.0,'score_H':1.5
    },
    {
        'nama': 'Andi Wijaya', 'umur': 30,
        'jenis_kelamin': 'Laki-laki', 'pendidikan': 'S2',
        'lama_hubungan': '3 - 5 tahun', 'lama_sejak_putus': '> 1 tahun',
        'siapa_yang_mengakhiri': 'Saya yang mengakhiri',
        'masih_komunikasi': 'Tidak pernah', 'frekuensi_medsos': 'Tidak pernah',
        'score_B':1.5,'score_C':4.5,'score_D':1.5,'score_E':1.5,'score_F':4.5,'score_G':1.5,'score_H':4.5
    },
    {
        'nama': 'Diana Putri', 'umur': 27,
        'jenis_kelamin': 'Perempuan', 'pendidikan': 'S1',
        'lama_hubungan': '1 - 3 tahun', 'lama_sejak_putus': '3 - 6 bulan',
        'siapa_yang_mengakhiri': 'Tidak jelas',
        'masih_komunikasi': 'Kadang-kadang', 'frekuensi_medsos': 'Hampir setiap hari',
        'score_B':3.5,'score_C':3.0,'score_D':3.5,'score_E':3.0,'score_F':3.0,'score_G':3.5,'score_H':3.0
    },
]

print('=' * 55)
print('  📊 BATCH PREDICTION RESULTS')
print('=' * 55)

rows = []
for case in test_cases:
    r = predict_heartbreak(
        umur=case['umur'], jenis_kelamin=case['jenis_kelamin'],
        pendidikan=case['pendidikan'], lama_hubungan=case['lama_hubungan'],
        sudah_berapa_lama_putus=case['lama_sejak_putus'],
        siapa_yang_mengakhiri=case['siapa_yang_mengakhiri'],
        masih_komunikasi=case['masih_komunikasi'],
        frekuensi_medsos_mantan=case['frekuensi_medsos'],
        score_B=case['score_B'], score_C=case['score_C'],
        score_D=case['score_D'], score_E=case['score_E'],
        score_F=case['score_F'], score_G=case['score_G'],
        score_H=case['score_H']
    )
    rows.append({
        'Nama':          case['nama'],
        'Umur':          case['umur'],
        'Jenis Kelamin': case['jenis_kelamin'],
        'Lama Hubungan': case['lama_hubungan'],
        'Sejak Putus':   case['lama_sejak_putus'],
        'Prediksi':      r['severity'],
        'Confidence':    f"{r['confidence']*100:.1f}%"
    })

summary_df = pd.DataFrame(rows)
print()
print(summary_df.to_string(index=False))
print()

  📊 BATCH PREDICTION RESULTS

       Nama  Umur Jenis Kelamin Lama Hubungan Sejak Putus Prediksi Confidence
Siti Rahayu    22     Perempuan     < 6 bulan   < 1 bulan   Sedang      93.1%
Andi Wijaya    30     Laki-laki   3 - 5 tahun   > 1 tahun   Ringan      98.0%
Diana Putri    27     Perempuan   1 - 3 tahun 3 - 6 bulan   Ringan     100.0%



In [23]:
# ============================================================
# CELL 5: DEBUG — TAMPILKAN 39 FEATURE NAMES
# ============================================================
print('📋 Feature Names (dari scaler — ground truth):')
print()
for i, feat in enumerate(TRUE_FEATURE_NAMES, 1):
    print(f'  {i:2d}. {feat}')

📋 Feature Names (dari scaler — ground truth):

   1. Umur
   2. Jenis Kelamin_Laki -  Laki
   3. Jenis Kelamin_Laki-laki
   4. Jenis Kelamin_Perempuan
   5. Pendidikan_Diploma (D1/D2/D3)
   6. Pendidikan_Lainnya
   7. Pendidikan_S1
   8. Pendidikan_S2
   9. Pendidikan_S3
  10. Pendidikan_SMA/Sederajat
  11. Pendidikan_SMP/Sederajat
  12. Lama Hubungan Sebelum Putus_3 - 5 tahun
  13. Lama Hubungan Sebelum Putus_6 bulan - 1 tahun
  14. Lama Hubungan Sebelum Putus_< 6 bulan
  15. Lama Hubungan Sebelum Putus_> 5 tahun
  16. Sudah Berapa Lama Sejak Putus?_3 - 6 bulan
  17. Sudah Berapa Lama Sejak Putus?_6 - 12 bulan
  18. Sudah Berapa Lama Sejak Putus?_< 1 bulan
  19. Sudah Berapa Lama Sejak Putus?_> 1 tahun
  20. Siapa yang Mengakhiri Hubungan?_Pasangan yang mengakhiri
  21. Siapa yang Mengakhiri Hubungan?_Saya yang mengakhiri
  22. Siapa yang Mengakhiri Hubungan?_Tidak jelas
  23. Apakah Masih Berkomunikasi dengan Mantan?_Kadang-kadang
  24. Apakah Masih Berkomunikasi dengan Mantan?_Serin